In [18]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Mon_Oct_24_19:12:58_PDT_2022
Cuda compilation tools, release 12.0, V12.0.76
Build cuda_12.0.r12.0/compiler.31968024_0


In [19]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Quadro RTX 5000


In [ ]:
%%writefile hello.cu
#include <stdio.h>

__global__ void helloKernel() {
  printf("hello from thread %d, block %d\n", threadIdx.x, blockIdx.x);
}

// general structure for a kernel:
// -> grid (1 per kernel)
//   -> blocks (65535 per dimension)
//   -> threads (1024 per block)
//   -> streaming multiprocessors(SM) (48)
//   -> warp size (32 threads per SM)

__device__ void printThread() { printf("thread: %d\n", threadIdx.x); }

void printFloatArray(float *arr, int n) {
  for (int i = 0; i < n; i++) {
    printf("array element %d: %f\n", i, arr[i]);
  }
}

// implement a kernel that adds 10 to each position of vector
// `a` and stores it in vector `out`. 1 thread per position.
__global__ void addTen(float *a, float *out) {
  printThread();
  out[threadIdx.x] = a[threadIdx.x] + 10;
}
void runAddTen() {
  float *d_a;
  float *d_out;
  float a[] = {1, 2, 3, 4, 5, 6, 7, 8};
  float out[8];
  cudaMalloc(&d_a, 8 * sizeof(float));
  cudaMalloc(&d_out, 8 * sizeof(float));
  cudaMemcpy(d_a, a, 8 * sizeof(float), cudaMemcpyHostToDevice);
  addTen<<<1, 8>>>(d_a, d_out);
  cudaMemcpy(out, d_out, 8 * sizeof(float), cudaMemcpyDeviceToHost);
  printf("back on device");
  printFloatArray(out, 8);
}

// a kernel that adds together each position of `a` and `b`
// and stores it in `out`. 1 thread per position.
__global__ void combineAB(float *a, float *b, float *out) {
  printThread();
  out[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
};

void runCombineAB() {
  float *d_a, *d_b;
  float *d_out;
  int len = 5;
  int size_f = len * sizeof(float);
  float a[] = {0, 1, 2, 3, 4};
  float b[] = {5, 6, 7, 8, 9};
  float out[len];
  cudaMalloc(&d_a, size_f);
  cudaMalloc(&d_b, size_f);
  cudaMalloc(&d_out, size_f);
  cudaMemcpy(d_a, a, size_f, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, size_f, cudaMemcpyHostToDevice);
  combineAB<<<1, len>>>(d_a, d_b, d_out);
  cudaMemcpy(out, d_out, size_f, cudaMemcpyDeviceToHost);
}
int main() {
  // float *d_a;
  // float *d_out;
  // float a[] = {1, 2, 3, 4, 5, 6, 7, 8};
  // float out[8];
  // cudaMalloc(&d_a, 8 * sizeof(float));
  // cudaMalloc(&d_out, 8 * sizeof(float));
  // cudaMemcpy(d_a, a, 8 * sizeof(float), cudaMemcpyHostToDevice);
  // addTen<<<1, 8>>>(d_a, d_out, 8);
  // cudaMemcpy(out, d_out, 8 * sizeof(float), cudaMemcpyDeviceToHost);
  // for (int i = 0; i < 8; i++) {
  //   printf("array element %d: %f\n", i, out[i]);
  // }
  // helloKernel<<<2, 4>>>();
  // helloKernel<<<1028, 4>>>();
  // runAddTen();
  runCombineAB();
  cudaDeviceSynchronize();
  return 0;
}

Overwriting hello.cu


In [23]:
!nvcc hello.cu -o hello

hello.cu(61): warning #549-D: variable "d_out" is used before its value is set

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

hello.cu(61): warning #549-D: variable "d_out" is used before its value is set

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"



In [17]:
!./hello

thread: 0
thread: 1
thread: 2
thread: 3
thread: 4
thread: 5
thread: 6
thread: 7
back on device
array element 0: 11.000000
array element 1: 12.000000
array element 2: 13.000000
array element 3: 14.000000
array element 4: 15.000000
array element 5: 16.000000
array element 6: 17.000000
array element 7: 18.000000
